In [38]:
import sys
print(sys.executable)

c:\Users\Rowan\anaconda3\python.exe


In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_PATH = "Most Streamed Spotify Songs 2024.csv"

In [40]:
df = pd.read_csv(RAW_PATH, encoding='latin1')

df.head()

,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",...,"2,182",264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",...,1,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1


## Attribute Overview & Mixed Data Types

A mix of data types:
- **Identifiers/text**: `Track`, `Album Name`, `Artist`, `ISRC`
- **Date**: `Release Date`
- **Binary/categorical**: `Explicit Track`
- **True numeric (float)**: `Track Score`, `Spotify Popularity`, `Apple Music Playlist Count`, `Deezer Playlist Count`, `Amazon Playlist Count`, `TIDAL Popularity`
- **Numeric-but-stored-as-text**: most streaming/engagement counts (`Spotify Streams`, `YouTube Views`, `TikTok Posts`, `Shazam Counts`, …) are stored as comma-formatted strings (e.g. `"390,470,936"`), so pandas reads them as `object`/string columns rather than numbers.
                                                                    

## Clean up the data.
Check for missing data.

Check percentage that data is missing,

Convert data with commas into numbers.

Check for duplicates

Remove Duplicates

In [53]:
df_clean =df.copy()

missing_data = df_clean.isnull().sum()
print(missing_data)

Track                            0
Album Name                       0
Artist                           5
Release Date                     0
ISRC                             0
All Time Rank                    0
Track Score                      0
Spotify Streams                113
Spotify Playlist Count          70
Spotify Playlist Reach          72
Spotify Popularity             804
YouTube Views                  308
YouTube Likes                  315
TikTok Posts                  1173
TikTok Likes                   980
TikTok Views                   981
YouTube Playlist Reach        1009
Apple Music Playlist Count     561
AirPlay Spins                  498
SiriusXM Spins                2123
Deezer Playlist Count          921
Deezer Playlist Reach          928
Amazon Playlist Count         1055
Pandora Streams               1106
Pandora Track Stations        1268
Soundcloud Streams            3333
Shazam Counts                  577
TIDAL Popularity              4600
Explicit Track      

In [54]:
missing_data_percentage = (missing_data / len(df) * 100).round(1)
print(missing_data_percentage,)

Track                           0.0
Album Name                      0.0
Artist                          0.1
Release Date                    0.0
ISRC                            0.0
All Time Rank                   0.0
Track Score                     0.0
Spotify Streams                 2.5
Spotify Playlist Count          1.5
Spotify Playlist Reach          1.6
Spotify Popularity             17.5
YouTube Views                   6.7
YouTube Likes                   6.8
TikTok Posts                   25.5
TikTok Likes                   21.3
TikTok Views                   21.3
YouTube Playlist Reach         21.9
Apple Music Playlist Count     12.2
AirPlay Spins                  10.8
SiriusXM Spins                 46.2
Deezer Playlist Count          20.0
Deezer Playlist Reach          20.2
Amazon Playlist Count          22.9
Pandora Streams                24.0
Pandora Track Stations         27.6
Soundcloud Streams             72.5
Shazam Counts                  12.5
TIDAL Popularity            

All of TIDAL popularity is missing. Drop it.

In [55]:
df_clean = df_clean.drop(columns=['TIDAL Popularity'], errors='ignore')

for c in df_clean.columns:
    print(c)

Track
Album Name
Artist
Release Date
ISRC
All Time Rank
Track Score
Spotify Streams
Spotify Playlist Count
Spotify Playlist Reach
Spotify Popularity
YouTube Views
YouTube Likes
TikTok Posts
TikTok Likes
TikTok Views
YouTube Playlist Reach
Apple Music Playlist Count
AirPlay Spins
SiriusXM Spins
Deezer Playlist Count
Deezer Playlist Reach
Amazon Playlist Count
Pandora Streams
Pandora Track Stations
Soundcloud Streams
Shazam Counts
Explicit Track


Check colums with strings
Convert to numbers

In [56]:
dtype_counts = df.dtypes.value_counts()
print("Column dtype counts as loaded:\n", dtype_counts, "\n")

numeric_as_text_columns = [c for c in df.columns
                           if not pd.api.types.is_numeric_dtype(df[c])
                            and df[c].dropna().astype(str).str.replace(',', '', regex=False).str.match(r'^-?\d+\.?\d*$').all()
                            and c not in ['ISRC']]

print((len(numeric_as_text_columns),"columns have commas"))

print('these columns have commas', numeric_as_text_columns)

Column dtype counts as loaded:
 object     22
float64     6
int64       1
Name: count, dtype: int64 

(17, 'columns have commas')
these columns have commas ['All Time Rank', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Reach', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts']


In [57]:
def clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False), errors='coerce')

for c in numeric_as_text_columns:
    df_clean[c] = clean_numeric(df_clean[c])

print(df_clean[numeric_as_text_columns].dtypes.value_counts())
df_clean[['Spotify Streams', 'YouTube Views', 'Shazam Counts']].head(5)

float64    16
int64       1
Name: count, dtype: int64


,Spotify Streams,YouTube Views,Shazam Counts
0,3.904709e+08,8.427475e+07,2669262.0
1,3.237039e+08,1.163470e+08,1118279.0
2,6.013093e+08,1.225991e+08,5285340.0
3,2.031281e+09,1.096101e+09,11822942.0
4,1.070349e+08,7.737396e+07,457017.0
